## Nuance Content Generator (Dyson Protocol)

Simulate forum activity by creating markdown posts, replies, and tag ratings using on-chain Storage.

- Users: `alice`, `bob`, `charlie`
- Active tags: `dashboard`, `foo`, `bar`, `baz`
- Markdown source: [Lorem Markdownum](https://jaspervdj.be/lorem-markdownum/markdown.txt)
- Storage owner: `nuance.dys`

What it does:
- Creates posts with a random subset of active tags
- Randomly replies to existing posts
- Randomly rates tags (up/down) on existing posts

References:
- See `docs/` for CLI patterns and examples (e.g., `docs/dyslang_guide.md`, `docs/storage_guide.md`).


In [ ]:
import json, os, shlex, uuid, random, time, textwrap
from datetime import datetime

try:
    import requests  # preferred for fetching markdown
except Exception:
    requests = None
import urllib.request

USERS = ["alice", "bob", "charlie"]
TAGS = ["dashboard", "foo", "bar", "baz"]
OWNER = os.environ.get("NUANCE_STORAGE_OWNER", "nuance.dys")
LOREM_URL = "https://jaspervdj.be/lorem-markdownum/markdown.txt"

print({"OWNER": OWNER, "USERS": USERS, "TAGS": TAGS})

lorem_text = ""
try:
    if requests is not None:
        resp = requests.get(LOREM_URL, timeout=10)
        resp.raise_for_status()
        lorem_text = resp.text
    else:
        with urllib.request.urlopen(LOREM_URL, timeout=10) as r:
            lorem_text = r.read().decode("utf-8", "ignore")
except Exception as e:
    # Let errors bubble in later cells if needed; keep empty string fallback
    print(f"Warning: failed to fetch lorem markdown: {e}")

# Pre-split content into paragraphs-ish blocks
blocks = [b.strip() for b in lorem_text.split("\n\n") if b.strip()]

random.seed()

def sample_markdown(min_len=200, max_len=800):
    """Return a slice of the lorem corpus within size bounds."""
    if not blocks:
        return "# Placeholder\n\nHello world."
    chosen = []
    total = 0
    # Random start index and accumulate until limits
    start = random.randrange(0, max(1, len(blocks)))
    i = start
    while total < min_len and len(chosen) < len(blocks):
        chosen.append(blocks[i % len(blocks)])
        total = sum(len(c) for c in chosen) + 2 * (len(chosen) - 1)
        i += 1
    text = "\n\n".join(chosen)
    if len(text) > max_len:
        text = text[:max_len].rsplit("\n", 1)[0] or text[:max_len]
    return text

def new_id(prefix="id"):
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


In [ ]:
import subprocess


def run_cmd(args):
    res = subprocess.run(args, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return res.stdout


def run_json_cmd(args):
    out = run_cmd(args)
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        # Surface output for debugging
        print(out)
        raise


def wait_tx(txhash: str) -> dict:
    return run_json_cmd(["dysond", "query", "wait-tx", txhash, "-o", "json"])  # returns dict


def send_tx(args) -> dict:
    """Run a transaction and wait for inclusion, returning the wait-tx JSON dict."""
    tx_out = run_json_cmd(["dysond", *args, "-y", "-o", "json"])  # should include .txhash or .height/code
    txhash = tx_out.get("txhash") or tx_out.get("hash") or tx_out.get("TXHASH")
    if not txhash:
        # Some commands may return a list or nested; last resort scan for hex-like hash
        if isinstance(tx_out, dict):
            for v in tx_out.values():
                if isinstance(v, str) and len(v) >= 64 and all(c in "0123456789ABCDEF" for c in v.upper()[:64]):
                    txhash = v
                    break
    if not txhash:
        raise RuntimeError(f"Cannot find txhash in response: {tx_out}")
    return wait_tx(txhash)


def storage_set(key: str, value_obj: dict, from_user: str) -> dict:
    val = json.dumps(value_obj, separators=(",", ":"))
    return send_tx([
        "tx", "storage", "set",
        "--owner", OWNER,
        "--key", key,
        "--value", val,
        "--from", from_user,
        "--gas", "auto",
    ])


def storage_get(key: str) -> dict:
    return run_json_cmd(["dysond", "query", "storage", "get", "--owner", OWNER, "--key", key, "-o", "json"])  # {"key":..., "value":...}


def storage_list(prefix: str, limit: int | None = None) -> dict:
    args = ["dysond", "query", "storage", "list", "--owner", OWNER, "--index-prefix", prefix, "-o", "json"]
    if limit is not None:
        args += ["--limit", str(limit)]
    return run_json_cmd(args)
